In [2]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier
from IPython.display import display

In [3]:
warnings.filterwarnings("ignore")

# =========================================================
# CONFIG
# =========================================================
DATA_PATH   = "processed/chennai_aqi_era5_hourly.parquet"
OUT_DIR     = "huber_processed_leakfree"
SPLIT_TIME  = pd.Timestamp("2025-01-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4

# Tuning: reduce these for quick debugging, restore for final paper runs
RUN_TUNED   = True
JSO_POP     = 12
JSO_ITERS   = 15
RANDOM_SEED = 1


In [4]:
# =========================================================
# HELPERS
# =========================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

def bearing_radians(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(y, x)

def flatten_window_per_node(X):
    S, Lx, N, F = X.shape
    return X.transpose(0, 2, 1, 3).reshape(S * N, Lx * F)

def compute_tail_metrics(y_true, y_pred, percentiles=(90, 95, 99)):
    rows = []
    for p in percentiles:
        thr = np.percentile(y_true, p)
        idx = y_true >= thr
        if idx.sum() == 0:
            rows.append((p, thr, np.nan, np.nan, 0))
        else:
            rows.append((
                p,
                thr,
                mean_absolute_error(y_true[idx], y_pred[idx]),
                np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])),
                int(idx.sum())
            ))
    return rows

def compute_mfb_nmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mfb = np.mean(2.0 * (y_pred - y_true) / (y_pred + y_true + 1e-8))
    nmse = np.sum((y_pred - y_true) ** 2) / (np.sum(y_pred * y_true) + 1e-8)
    return mfb, nmse

def build_fill_values_from_train_timeline(X_train_timeline):
    # X_train_timeline: [T_train, N, F]
    fill_values = np.nanmedian(X_train_timeline, axis=0)  # [N, F]
    global_fill = np.nanmedian(X_train_timeline.reshape(-1, X_train_timeline.shape[-1]), axis=0)  # [F]
    global_fill = np.where(np.isnan(global_fill), 0.0, global_fill)
    for n in range(fill_values.shape[0]):
        for f in range(fill_values.shape[1]):
            if np.isnan(fill_values[n, f]):
                fill_values[n, f] = global_fill[f]
    fill_values = np.where(np.isnan(fill_values), 0.0, fill_values).astype(np.float32)
    return fill_values

def impute_windows(X, fill_values):
    # Strictly leakage-safe: operate within each window only, then fall back to TRAIN-only fill_values.
    X_imp = X.copy().astype(np.float32)
    S, Lx, N, F = X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                series = pd.Series(X_imp[s, :, n, f], dtype="float32")
                series = series.ffill().bfill()  # only inside the current historical window
                arr = series.to_numpy(dtype=np.float32)
                if np.isnan(arr).any():
                    arr = np.where(np.isnan(arr), fill_values[n, f], arr)
                X_imp[s, :, n, f] = arr
    return X_imp

def build_nodes_edges(df, station_ids, k=4):
    coords = (
        df[["station_id", "lat", "lon"]]
        .drop_duplicates()
        .assign(station_id=lambda d: d["station_id"].astype(str))
    )
    nodes = coords[coords["station_id"].isin(station_ids)].copy()
    nodes = nodes.set_index("station_id").loc[station_ids].reset_index()
    nodes["node_id"] = np.arange(len(nodes))

    coords_arr = nodes[["lat", "lon"]].to_numpy(dtype=float)
    N = len(nodes)

    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords_arr[i, 0], coords_arr[i, 1], coords_arr[:, 0], coords_arr[:, 1])

    sigma = np.median(D[D > 0])
    edges = []
    kk = min(k, N - 1)
    for i in range(N):
        nn = np.argsort(D[i])[1:kk + 1]
        for j in nn:
            w = np.exp(-(D[i, j] ** 2) / (2.0 * sigma ** 2))
            edges.append((i, j, float(w), float(D[i, j])))

    edges_df = pd.DataFrame(edges, columns=["src", "dst", "w_dist", "dist_km"])
    return nodes, edges_df

def build_static_adj_from_nodes(nodes_df, k=4, sigma_km=None, self_loops=False, row_normalize=True):
    coords = nodes_df[["lat", "lon"]].to_numpy(dtype=float)
    N = coords.shape[0]
    D = np.zeros((N, N), dtype=float)
    for i in range(N):
        D[i, :] = haversine_km_vec(coords[i, 0], coords[i, 1], coords[:, 0], coords[:, 1])

    if sigma_km is None:
        sigma_km = np.median(D[D > 0])

    A = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        nn = np.argsort(D[i])[1:min(k + 1, N)]
        for j in nn:
            A[i, j] = np.float32(np.exp(-(D[i, j] ** 2) / (2.0 * sigma_km ** 2)))

    if self_loops:
        np.fill_diagonal(A, 1.0)

    if row_normalize:
        rs = A.sum(axis=1, keepdims=True)
        A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)

    return A

def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05):
    theta_w = np.arctan2(v_t[ctx["src"]], u_t[ctx["src"]])  # wind direction at src nodes
    align = np.cos(theta_w - ctx["edge_bearing"])           # directional alignment with edge
    gate = 1.0 / (1.0 + np.exp(-alpha * align))
    gate = eps + (1.0 - eps) * gate
    w_dyn = ctx["w_dist"] * gate

    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[ctx["src"], ctx["dst"]] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    return A

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        A = build_dynamic_adj(
            x_now[:, ctx["u_idx"]],
            x_now[:, ctx["v_idx"]],
            ctx,
            alpha=alpha,
            eps=eps
        )
        agg = A @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def make_graph_features_static(X, A_static, tau=0):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    tau = int(tau)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        agg = A_static @ x_lag
        Z[s] = np.concatenate([x_now, agg], axis=-1)
    return Z

def summarize_regression(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred))
    }

def train_val_split_time_order(X_train_imp, Y_train, M_train, target_times_train, frac=0.2):
    n = X_train_imp.shape[0]
    val_size = max(1, int(np.ceil(frac * n)))
    idx_train = np.arange(0, n - val_size)
    idx_val   = np.arange(n - val_size, n)
    train_pack = {
        "X": X_train_imp[idx_train],
        "Y": Y_train[idx_train],
        "M": M_train[idx_train],
        "times": target_times_train[idx_train]
    }
    val_pack = {
        "X": X_train_imp[idx_val],
        "Y": Y_train[idx_val],
        "M": M_train[idx_val],
        "times": target_times_train[idx_val]
    }
    return train_pack, val_pack

def fit_flat_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test):
    Xtr = flatten_window_per_node(X_train_imp)
    Xte = flatten_window_per_node(X_test_imp)
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_static_graph_regressor(model, X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, A_static, tau):
    Ztr = make_graph_features_static(X_train_imp, A_static, tau=tau)
    Zte = make_graph_features_static(X_test_imp, A_static, tau=tau)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    model.fit(Xtr[mtr], ytr[mtr])
    pred_all = model.predict(Xte)
    return {
        "y_true": yte[mte],
        "y_pred": pred_all[mte],
        "model": model
    }

def fit_huber_graph(X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test, ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4):
    Ztr = make_graph_features_dynamic(X_train_imp, ctx, tau=tau, alpha=alpha, eps=eps)
    Zte = make_graph_features_dynamic(X_test_imp, ctx, tau=tau, alpha=alpha, eps=eps)

    Xtr = Ztr.reshape(-1, Ztr.shape[-1])
    Xte = Zte.reshape(-1, Zte.shape[-1])
    ytr = Y_train.reshape(-1)
    yte = Y_test.reshape(-1)
    mtr = (M_train.reshape(-1) > 0.5)
    mte = (M_test.reshape(-1) > 0.5)

    huber = Pipeline([
        ("scaler", StandardScaler()),
        ("huber", HuberRegressor(epsilon=1.35, alpha=huber_alpha, max_iter=500))
    ])
    huber.fit(Xtr[mtr], ytr[mtr])

    pred_train_all = huber.predict(Xtr)
    pred_test_all  = huber.predict(Xte)

    return {
        "y_true": yte[mte],
        "y_pred": pred_test_all[mte],
        "pred_train_all": pred_train_all,
        "pred_test_all": pred_test_all,
        "y_train_all": ytr,
        "y_test_all": yte,
        "mask_train": mtr,
        "mask_test": mte,
        "model": huber
    }

def fit_huber_hybrid(
    X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
    ctx, tau=0, alpha=4.0, eps=0.05, huber_alpha=1e-4,
    xgb_params=None, num_boost_round=400, seed=42,
    w_mid=2.0, w_danger=5.0, w_tail=10.0
):
    base = fit_huber_graph(
        X_train_imp, Y_train, M_train, X_test_imp, Y_test, M_test,
        ctx=ctx, tau=tau, alpha=alpha, eps=eps, huber_alpha=huber_alpha
    )

    ytr_all = base["y_train_all"]
    pred_tr_all = base["pred_train_all"]
    mtr = base["mask_train"]
    mte = base["mask_test"]

    res_train = ytr_all - pred_tr_all
    Xgb_train = np.asarray(flatten_window_per_node(X_train_imp), dtype=np.float32)
    Xgb_test  = np.asarray(flatten_window_per_node(X_test_imp),  dtype=np.float32)

    t90 = np.percentile(ytr_all[mtr], 90)
    t95 = np.percentile(ytr_all[mtr], 95)
    t99 = np.percentile(ytr_all[mtr], 99)

    weights = np.ones_like(ytr_all[mtr], dtype=np.float32)
    weights[ytr_all[mtr] >= t90] = w_mid
    weights[ytr_all[mtr] >= t95] = w_danger
    weights[ytr_all[mtr] >= t99] = w_tail

    dtrain = xgb.DMatrix(Xgb_train[mtr], label=res_train[mtr], weight=weights)

    if xgb_params is None:
        xgb_params = {
            "objective": "reg:pseudohubererror",
            "max_depth": 6,
            "eta": 0.05,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "lambda": 1.0,
            "tree_method": "hist",
            "seed": seed,
            "verbosity": 0,
        }

    booster = xgb.train(xgb_params, dtrain, num_boost_round=int(num_boost_round))

    res_test_all = booster.predict(xgb.DMatrix(Xgb_test))
    final_test_all = base["pred_test_all"].copy()
    final_test_all[mte] = final_test_all[mte] + res_test_all[mte]

    return {
        "y_true": base["y_test_all"][mte],
        "y_pred_graph": base["pred_test_all"][mte],
        "y_pred_final": final_test_all[mte],
        "gmodel": base["model"],
        "hmodel": booster
    }

def choose_best_tau_dynamic(train_pack, val_pack, ctx, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        res = fit_huber_graph(
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            ctx=ctx, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(train_pack, val_pack, A_static, tau_list):
    rows = []
    best_tau = None
    best_mae = np.inf
    for tau in tau_list:
        if tau >= train_pack["X"].shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("huber", HuberRegressor(epsilon=1.35, max_iter=500))
        ])
        res = fit_static_graph_regressor(
            model,
            train_pack["X"], train_pack["Y"], train_pack["M"],
            val_pack["X"], val_pack["Y"], val_pack["M"],
            A_static=A_static, tau=tau
        )
        mae = mean_absolute_error(res["y_true"], res["y_pred"])
        rows.append({"tau": tau, "val_MAE": mae})
        if mae < best_mae:
            best_mae = mae
            best_tau = tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params, tau, train_pack, val_pack, ctx):
    alpha, eps, log_huber_alpha, max_depth, eta, subsample, colsample, reg_lambda, nround, w_mid, w_danger, w_tail = params
    huber_alpha = 10 ** float(log_huber_alpha)

    hybrid = fit_huber_hybrid(
        train_pack["X"], train_pack["Y"], train_pack["M"],
        val_pack["X"],   val_pack["Y"],   val_pack["M"],
        ctx=ctx,
        tau=tau,
        alpha=float(alpha),
        eps=float(eps),
        huber_alpha=float(huber_alpha),
        xgb_params={
            "objective": "reg:pseudohubererror",
            "max_depth": int(round(max_depth)),
            "eta": float(eta),
            "subsample": float(subsample),
            "colsample_bytree": float(colsample),
            "lambda": float(reg_lambda),
            "tree_method": "hist",
            "seed": RANDOM_SEED,
            "verbosity": 0,
        },
        num_boost_round=int(round(nround)),
        seed=RANDOM_SEED,
        w_mid=float(w_mid),
        w_danger=float(w_danger),
        w_tail=float(w_tail)
    )

    y_true = hybrid["y_true"]
    y_pred = hybrid["y_pred_final"]
    mae = mean_absolute_error(y_true, y_pred)

    thr99 = np.percentile(y_true, 99)
    idx_tail = y_true >= thr99
    if idx_tail.sum() > 0:
        mfb_tail, _ = compute_mfb_nmse(y_true[idx_tail], y_pred[idx_tail])
        return mae + (10.0 * abs(mfb_tail))
    return mae

def sample_population(n, bounds):
    pop = np.random.rand(n, bounds.shape[0])
    return bounds[:, 0] + pop * (bounds[:, 1] - bounds[:, 0])

def clip_params(x, bounds):
    return np.minimum(np.maximum(x, bounds[:, 0]), bounds[:, 1])

def jso_optimize(train_pack, val_pack, ctx, tau, bounds, n_pop=12, iters=15, seed=1):
    np.random.seed(seed)
    dim = bounds.shape[0]
    pop = sample_population(n_pop, bounds)
    fitness = np.array([eval_hybrid_params(p, tau, train_pack, val_pack, ctx) for p in pop], dtype=float)

    best_idx = np.argmin(fitness)
    best_p = pop[best_idx].copy()
    best_f = float(fitness[best_idx])
    print(f"    initial best objective={best_f:.4f}")

    for it in range(iters):
        c = (1.0 - it / max(iters, 1))
        new_pop = pop.copy()
        for i in range(n_pop):
            if np.random.rand() < 0.5:
                step = np.random.randn(dim) * c * 0.1
                cand = pop[i] + step + c * (best_p - pop[i]) * np.random.rand(dim)
            else:
                j = np.random.randint(0, n_pop)
                step = (pop[j] - pop[i]) * (np.random.rand(dim) - 0.5) * c
                cand = pop[i] + step
            new_pop[i] = clip_params(cand, bounds)

        new_fit = np.array([eval_hybrid_params(p, tau, train_pack, val_pack, ctx) for p in new_pop], dtype=float)
        improved = new_fit < fitness
        pop[improved] = new_pop[improved]
        fitness[improved] = new_fit[improved]

        best_idx = np.argmin(fitness)
        if fitness[best_idx] < best_f:
            best_f = float(fitness[best_idx])
            best_p = pop[best_idx].copy()

        print(f"    iter {it + 1:02d}/{iters:02d} -> best objective={best_f:.4f}")

    return best_p, best_f

def add_result_row(rows, tail_rows, H, split_name, model_name, y_true, y_pred, extra=None):
    summary = summarize_regression(y_true, y_pred)
    row = {
        "H": H,
        "split": split_name,
        "model": model_name,
        "MAE": summary["MAE"],
        "RMSE": summary["RMSE"],
        "R2": summary["R2"],
        "n_test": int(len(y_true))
    }
    if extra:
        row.update(extra)
    rows.append(row)

    for p, thr, mae, rmse, n_tail in compute_tail_metrics(y_true, y_pred):
        tail_row = {
            "H": H,
            "split": split_name,
            "model": model_name,
            "percentile": p,
            "threshold": float(thr),
            "tail_MAE": float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE": float(rmse) if pd.notna(rmse) else np.nan,
            "n_tail": int(n_tail)
        }
        if extra:
            tail_row.update(extra)
        tail_rows.append(tail_row)


In [5]:
# =========================================================
# 1) BUILD LEAKAGE-FREE RAW TENSOR
# =========================================================
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH).copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["station_id"] = df["station_id"].astype(str)

stations = (
    df[["station_id", "station_name"]]
    .drop_duplicates()
    .sort_values("station_id")
    .reset_index(drop=True)
)
station_ids = stations["station_id"].tolist()
N = len(station_ids)

candidate_features = ["pm25", "temp_2m", "dewpoint_2m", "surface_pressure", "u10", "v10"]
features = [c for c in candidate_features if c in df.columns]
assert "pm25" in features, "pm25 column not found in parquet."
pm25_idx = features.index("pm25")
u_idx = features.index("u10")
v_idx = features.index("v10")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids], names=["timestamp", "station_id"]).to_frame(index=False)

aligned = base.merge(
    df[["timestamp", "station_id"] + features],
    on=["timestamp", "station_id"],
    how="left"
)

X_feat = []
for feat in features:
    mat = aligned.pivot(index="timestamp", columns="station_id", values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))

X_all_raw = np.stack(X_feat, axis=-1).astype(np.float32)   # [T, N, F]

pm25_raw = aligned.pivot(index="timestamp", columns="station_id", values="pm25").reindex(all_times)[station_ids]
pm25_raw_np = pm25_raw.to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(pm25_raw_np)).astype(np.float32)

# Fill values are learned from TRAIN timeline only -> no leakage from future/test period
train_timeline_mask = all_times < SPLIT_TIME
fill_values = build_fill_values_from_train_timeline(X_all_raw[train_timeline_mask])

# =========================================================
# 2) BUILD GRAPH STRUCTURE ONCE
# =========================================================
nodes, edges_df = build_nodes_edges(df, station_ids, k=GRAPH_K)
A_static = build_static_adj_from_nodes(nodes, k=GRAPH_K, sigma_km=None, self_loops=False, row_normalize=True)

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)

src_lat = nodes.loc[src, "lat"].to_numpy()
src_lon = nodes.loc[src, "lon"].to_numpy()
dst_lat = nodes.loc[dst, "lat"].to_numpy()
dst_lon = nodes.loc[dst, "lon"].to_numpy()
edge_bearing = bearing_radians(src_lat, src_lon, dst_lat, dst_lon)

graph_ctx = {
    "src": src,
    "dst": dst,
    "w_dist": w_dist,
    "edge_bearing": edge_bearing,
    "u_idx": u_idx,
    "v_idx": v_idx,
    "N": len(nodes),
}

In [6]:
from scipy.stats import wilcoxon

print("Running Wilcoxon Signed-Rank Test (Tuned-Hybrid vs XGB) on 99th Percentile Tail...")
print("=" * 85)

# 1. Load your saved models from the dead kernel session
all_models = joblib.load(os.path.join(OUT_DIR, "all_models.pkl"))

results = []

for H in H_LIST:
    # 2. Load the exact test arrays generated for this horizon
    out_h = os.path.join(OUT_DIR, f"H{H}")
    X_test_imp = np.load(os.path.join(out_h, "X_test_imp.npy"))
    Y_test = np.load(os.path.join(out_h, "Y_test.npy"))
    M_test = np.load(os.path.join(out_h, "M_test.npy"))

    # Flatten test masks and targets to match your evaluation logic
    yte = Y_test.reshape(-1)
    mte = (M_test.reshape(-1) > 0.5)
    y_true = yte[mte]

    # --- A. RECONSTRUCT XGBOOST PREDICTIONS ---
    xgb_model = all_models[H]["xgb"]
    Xte_flat = flatten_window_per_node(X_test_imp)
    y_pred_xg = xgb_model.predict(Xte_flat)[mte]

    # --- B. RECONSTRUCT TUNED HYBRID PREDICTIONS ---
    gmodel, hmodel = all_models[H]["tuned_hybrid_graph"]
    tau = all_models[H]["selected_tau_dynamic"]

    # Rebuild graph features for the test set using your saved tau and graph_ctx
    Zte = make_graph_features_dynamic(X_test_imp, graph_ctx, tau=tau, alpha=4.0, eps=0.05)
    Xte_graph = Zte.reshape(-1, Zte.shape[-1])

    # Graph base prediction + XGB residual prediction
    pred_graph = gmodel.predict(Xte_graph)
    pred_res = hmodel.predict(xgb.DMatrix(Xte_flat))
    y_pred_th = (pred_graph + pred_res)[mte]

    # --- C. WILCOXON SIGNED-RANK TEST (99th Percentile Tail) ---
    thr99 = np.percentile(y_true, 99)
    idx_tail = y_true >= thr99

    err_th = np.abs(y_true[idx_tail] - y_pred_th[idx_tail])
    err_xg = np.abs(y_true[idx_tail] - y_pred_xg[idx_tail])

    # alternative='less' tests if the Hybrid error is significantly LESS than the XGB error
    stat_w, p_val = wilcoxon(err_th, err_xg, alternative='less')
    
    is_sig = p_val < 0.05

    results.append({
        'H': H,
        'TH_tail_MAE': err_th.mean(),
        'XGB_tail_MAE': err_xg.mean(),
        'p_value': p_val,
        'significant': is_sig
    })

    print(f"H={H:2d}h | n_tail={idx_tail.sum():3d} | "
          f"TH_tail_MAE={err_th.mean():.3f} | XGB_tail_MAE={err_xg.mean():.3f} | "
          f"p={p_val:.4e} | {'✓ SIGNIFICANT' if is_sig else '✗ NOT significant'}")

# Optional: Save these statistical results out to your directory
pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, "wilcoxon_significance.csv"), index=False)

Running Wilcoxon Signed-Rank Test (Tuned-Hybrid vs XGB) on 99th Percentile Tail...
H= 1h | n_tail=230 | TH_tail_MAE=46.458 | XGB_tail_MAE=50.446 | p=5.1587e-12 | ✓ SIGNIFICANT
H= 3h | n_tail=230 | TH_tail_MAE=53.634 | XGB_tail_MAE=56.894 | p=5.3014e-06 | ✓ SIGNIFICANT
H= 6h | n_tail=230 | TH_tail_MAE=55.391 | XGB_tail_MAE=58.660 | p=6.0526e-07 | ✓ SIGNIFICANT
H=12h | n_tail=230 | TH_tail_MAE=55.721 | XGB_tail_MAE=59.367 | p=7.8109e-10 | ✓ SIGNIFICANT
H=24h | n_tail=230 | TH_tail_MAE=53.775 | XGB_tail_MAE=59.444 | p=3.7033e-21 | ✓ SIGNIFICANT
